In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import glob
PAIRS_SHOW = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
parts = [pd.read_csv(f) for f in sorted(glob.glob(f"{RES_DIR}/coexpression/coexpr_cells_*.csv"))]
c = pd.concat(parts)
c = c[c["double_high_subclass"]]
regs = [r for r in FOCUS_ROIS if r in set(parts and pd.concat(parts)["roi"])]
rcol = {r: plt.get_cmap("plasma")(0.08 + 0.8 * i / max(1, len(regs) - 1)) for i, r in enumerate(regs)}
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5), sharey=True)
rng = np.random.default_rng(0)
for ax, (A, B) in zip(axes, PAIRS_SHOW):
    x = c[(c["gene_a"] == A) & (c["gene_b"] == B)]
    for i, r in enumerate(regs):
        xr = x[x["roi"] == r]
        if len(xr) == 0:
            ax.text(i, 0.97, "none", ha="center", va="bottom", fontsize=8, color="0.4", rotation=90)
            continue
        ax.scatter(i + rng.uniform(-0.18, 0.18, len(xr)), xr["obs_over_expected"], s=np.clip(xr["n_cells"] / 8, 25, 200), color=rcol[r], edgecolor="k", linewidth=0.4, zorder=3)
        m = xr["obs_over_expected"].median()
        ax.hlines(m, i - 0.3, i + 0.3, color="k", lw=2, zorder=4)
    ax.axhline(1, color="0.5", ls="--", lw=1)
    ax.set_xticks(range(len(regs)))
    ax.set_xticklabels(regs, rotation=90, fontsize=9)
    ax.set_title(f"{A} and {B}", fontsize=12)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("cells with both receptors high:\nobserved / expected if independent", fontsize=10)
fig.suptitle("Do the same neurons carry both receptors? Each dot = one subclass that is high in both receptors (dot size = cells); black bar = median per region\nabove the dashed line (1) = receptors co-occur in the same cells more than chance; 'high' = top quartile of the region's neurons (Allen WMB, 10x)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.9])
fig.savefig(f"{FIG_DIR}/ALL_coexpr_obs_vs_expected.png", dpi=170, bbox_inches="tight")
plt.show()